In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from ftfy import fix_text
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler

df = pd.read_csv(
    r'/Users/urielle.zach/Data-Preprocessing/CSVs/globalmart_dirty_orders_2000.csv', 
    keep_default_na=False,
    na_values=['-', '#N/A', 'N/A', 'NULL', '?', 'unknown', 'NONE', 'na', 'None', 'none', 'NaN', 'NA', 'nan', ''], # catch all null values before parsing
    encoding='utf-8'
)   

df.index = df.index + 2 ## shifted index to align with csv

columns = [
    'record_id', 'source_system', 'customer_id', 'customer_name', 
    'customer_email', 'phone_raw', 'age_raw', 'gender_raw', 
    'signup_date_raw', 'order_id', 'order_date_raw', 'ship_date_raw', 
    'delivery_date_raw', 'customer_timezone', 'country_raw', 'city_raw', 
    'postal_code_raw', 'product_category_raw', 'product_name_raw', 
    'sku_raw', 'quantity_raw', 'unit_price_raw', 'currency_raw', 
    'discount_raw', 'shipping_cost_raw', 'item_weight_raw', 
    'payment_method_raw', 'order_status_raw', 'returned_flag_raw', 
    'return_reason_raw', 'satisfaction_score_raw', 'loyalty_points_raw', 
    'site_visits_last30_raw', 'support_tickets_90d_raw', 
    'distance_to_warehouse_km_raw', 'campaign_source_raw', 
    'customer_segment_raw', 'support_ticket_date_raw', 
    'complaint_text_raw', 'ingestion_batch', 'data_source_note'
]

In [3]:
## PART 1
def returnRows(df):
    return f"Total row count: {len(df)}"

def returnDatatype(df):
    return df.dtypes

def checkMissingValues(df, columns):
    null_rows = df[columns].isnull().sum() ## returns how many null rows are there in the series
    missing_values = null_rows[null_rows > 0] ## filters all boolean values that are greater than 0 to be passed through {missing_values} variable
    total_rows = len(df)
    
    percentages = (missing_values / total_rows) * 100
    print("--------------------Missing Rows--------------------")
    summary_df = pd.DataFrame({
        'Missing Count': missing_values,
        'Percentage (%)': percentages.round(2)  
    })

    return summary_df

## specifies which columns and row has null values


## if {value} of {column_name} HAS duplicates:
    ## do: 
        ## return the row where the {value} is duplicated
        ## filter all {value} to show all true
        ## show the number of duplicated {value} in a column

def checkSpecificColumnDuplicates(df, column_name):
    duplicate_mask = df.duplicated(subset=[column_name], keep=False) ## checks every value in index to check if {value} has duplicates
    duplicated_rows = df[duplicate_mask] ## passes all duplicates into duplicated_rows
    duplicated_rows = duplicated_rows.dropna(subset=[column_name]) ## drops all values containing {NaN}
    duplicates_clean_view = duplicated_rows[['record_id', column_name]]

    print(f"Total amount of duplicated rows: {len(duplicates_clean_view)}")
    return duplicates_clean_view




def detectInvalidAge(df):
    age_numeric = pd.to_numeric(df["age_raw"], errors='coerce')
    invalid_condition = (age_numeric >= 100) | (age_numeric <= 13) | (age_numeric.isna())
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'age_clean']]

def detectInvalidDiscount(df):
    if df["discount_raw"].dtype == '0':
        mapping = {
        'freebie': '100%',
        'ten percent': '10%'
        }
            
        df["discount_raw"] = df["discount_raw"].replace(mapping, regex=True)
        df["discount_raw"] = df["discount_raw"].str.rstrip('%').astype(float)
        
    discounts = pd.to_numeric(df["discount_raw"], errors='coerce')    
    invalid_condition = (discounts <= 0) | (discounts >= 101) | (discounts.isna())
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'discount_raw']]

def invalid_delivery_sequence_flag(df):
    order_date = pd.to_datetime(df["order_date_raw"], format='mixed', errors='coerce')
    delivery_date_clean = pd.to_datetime(df["delivery_date_raw"], format='mixed', errors='coerce')
    invalid_condition = (order_date > delivery_date_clean)
    invalid_rows = df[invalid_condition]
    return invalid_rows[['record_id', 'order_date_raw', 'delivery_date_raw']]



## email validation regex: ^[\w-\.]+@([\w-]+\.)+[\w-]{2,4}$     

In [ ]:
### DATA CLEANING ####

## pretend record_id is here
def fixSource(df): # source system
    clean_source = df["source_system"].str.upper().str.strip()
    mapping = {
        r'(?i)^(STORE|IN-STORE)$'                       : 'In-Store',
        r'(?i)^(MOBILE-APP|MOBILE APP|MOBILE|APP)$'     : 'Mobile App',
        r'(?i)^(CALL CENTRE|CALL CENTER)$'              : 'Call Center',
        r'(?i)^(PARTNER-API|PARTNER API)$'              : 'Partner-API',
        r'(?i)^(MARKETPLACE|MARKET PLACE)$'             : 'Marketplace',
        r'MARKETPLACE | MARKET PLACE'                   : 'Marketplace',
        r'(?i)^WEB$'                                    : 'Web'
    }
    
    print(f"dict: {mapping} \n")
    df["source_system_clean"] = clean_source.replace(mapping, regex=True)
    return df[['source_system', 'source_system_clean']]



# TODO
def fixCustomerID(df): ## checks if values have null.
    return 0




def fixEncoding(df):
   df['customer_name_clean'] = df['customer_name'].apply(lambda x: fix_text(x) if isinstance(x, str) else x)
   return df[['customer_name', 'customer_name_clean']]



# TODO
def fixPhone(df):
    return 0



def fixAge(df):
    ##FIXME: 
    df['clean_age'] = pd.to_numeric(df["age_raw"], errors='coerce')
    return df[['age_raw', 'clean_age']]



def fixGender(df): #gender_raw
    clean_source = df["gender_raw"].str.upper().str.strip()
    mapping = {
            r'(?i)^(NON-BINARY|NB)$'        : 'Non-Binary', 
            r'(?i)^(MALE|M)$'               : 'Male', 
            r'(?i)^(FEMALE|F)$'             :'Female', 
            r'(?i)^PREFER NOT TO SAY$'      : 'Prefer not to say'
    }
    
    print(f"dict: {mapping} \n")
    df['gender_clean'] = clean_source.replace(mapping, regex=True)
    print("remapped gender_raw")
    return df[['gender_raw', 'gender_clean']]



def fixTime(df):
   date_columns = ['signup_date_raw', 'order_date_raw', 'ship_date_raw', 'delivery_date_raw', 'support_ticket_date_raw']
   for i in date_columns:
      parsed_dates = pd.to_datetime(df[i], format="mixed", errors="coerce")
      df[i] = parsed_dates.dt.strftime('%Y-%m-%d')



# TODO
def fixTimeZone(df):
    return 0



def fixCountry(df): ## did this part earlier to fix all data
    clean_source = df["country_raw"].str.strip()
    mapping = {
    r'(?i)^(AUS?|Australia)$'                           : 'Australia',
    r'(?i)^(DE|Ger(many)?|Deutschland)$'                : 'Germany',
    r'(?i)^(FR(ance|nce)?)$'                            : 'France',
    r'(?i)^(SG|Singapore|S\'pore)$'                     : 'Singapore',
    r'(?i)^(JP|Japan|Nippon)$'                          : 'Japan',
    r'(?i)^(CA(nada|nda)?)$'                            : 'Canada',
    r'(?i)^(BR(asil|azil)?)$'                           : 'Brazil',
    r'(?i)^(PHL?|Phils\.?|Philippines|Pilipinas)$'      : 'Philippines',
    r'(?i)^(USA|U\.S\.A\.|US|United States|America)$'   : 'United States',
    r'(?i)^(CH|Swi(tzerland|ss)|Suisse)$'               : 'Switzerland'
    }
    
    print(f"dict: {mapping} \n")
    df["country_clean"] = clean_source.replace(mapping, regex=True)
    return df[['country_raw', 'country_clean']]



# TODO
def fixPostal(df):
    return 0



def fixCategory(df):
    clean_source = df["product_category_raw"].str.upper().str.strip()
    mapping = {
        r'^ELEC.*'                  : 'Electronics', 
        r'^(FASH|APP|CLOTH).*'      : 'Fashion & Apparel', 
        r'.*(HOME|KITCHEN).*'       : 'Home & Kitchen', 
        r'.*(BEAUTY|CARE).*'        : 'Health & Beauty', 
        r'.*(BOOK|READ|MEDIA).*'    : 'Books & Media', 
        r'.*(SPORT|FITNESS).*'      : 'Sports & Fitness'
    }

    print(f"dict: {mapping} \n")
    df["product_category_clean"] = clean_source.replace(mapping, regex=True)
    return df[['product_category_raw', 'product_category_clean']]



def fixQuantity(df):
    word_mapping = {
        'one'       : '1',
        'two'       : '2', 
        'three'     : '3', 
        'four'      : '4', 
        'five'      : '5'
    }

    cleaned_text = df["quantity_raw"].replace(word_mapping)
    extracted_numbers = cleaned_text.astype(str).str.extract(r'(-?\d+)', expand=False)
    quantity = pd.to_numeric(extracted_numbers, errors='coerce')
    invalid_condition = (quantity <= 0) | (quantity >= 250)

    df["quantity_clean"] = np.where(invalid_condition, np.nan, quantity)

    return df[['quantity_raw', 'quantity_clean']]



def fixUnitPrice(df):
    cleaned = df["unit_price_raw"].astype(str).str.replace(r'[^\d\.,]', '', regex=True)
    cleaned = cleaned.str.replace(r',(\d{1,2})$', r'.\1', regex=True)
    cleaned = cleaned.str.replace(',', '', regex=True)
    unit_price = pd.to_numeric(cleaned, errors='coerce')
    df["unit_price_clean"] = np.where(unit_price <= 0, np.nan, unit_price)
    
    return df[['unit_price_raw', 'unit_price_clean']]



def fixCurrency(df):
    return 0



def fixDiscount(df):
    cleaned_text = df["discount_raw"].astype(str).str.lower()
    mapping = {
        'freebie': '100',
        'ten percent': '10'
    }
    cleaned_text = cleaned_text.replace(mapping, regex=True)
    cleaned_text = cleaned_text.str.replace(r'[^\d\.]', '', regex=True)
    discounts = pd.to_numeric(cleaned_text, errors='coerce')    
    invalid_condition = (discounts < 0) | (discounts > 100)
    df["discount_clean"] = np.where(invalid_condition, np.nan, discounts)
    
    return df[['discount_raw', 'discount_clean']]



def fixShippingCost(df):
    cleaned = df["shipping_cost_raw"].astype(str).str.replace(r'[^\d\.,]', '', regex=True)
    cleaned = cleaned.str.replace(r',(\d{1,2})$', r'.\1', regex=True)
    cleaned = cleaned.str.replace(',', '', regex=True)
    unit_price = pd.to_numeric(cleaned, errors='coerce')
    df["shipping_cost_clean"] = np.where(unit_price <= 0, np.nan, unit_price)
    
    return df[['shipping_cost_raw', 'shipping_cost_clean']]



def fixItemWeight(df):
    is_lbs = df["item_weight_raw"].astype(str).str.lower().str.contains('lb', na=False)
    cleaned = df["item_weight_raw"].astype(str).str.replace(r'[^\d\.,]', '', regex=True)
    cleaned = cleaned.str.replace(r',(\d{1,2})$', r'.\1', regex=True)
    cleaned = cleaned.str.replace(',', '', regex=True)
    
    weight = pd.to_numeric(cleaned, errors='coerce')
    weight_in_kg = np.where(is_lbs, weight * 0.453592, weight)
    df["item_weight_clean"] = np.where(weight_in_kg <= 0, np.nan, weight_in_kg)
    
    return df[['item_weight_raw', 'item_weight_clean']]



def fixPaymentMethod(df):
    clean_source = df["payment_method_raw"].astype(str).str.upper().str.strip()
    mapping = {
        r'.*PAY\s*PAL.*'                            : 'PayPal', 
        r'.*APPLE.*'                                : 'Apple Pay', 
        r'.*G.*CASH.*'                              : 'GCash', 
        r'.*(COD|CASH.*DELIV).*'                    : 'Cash on Delivery', 
        r'.*(BANK|WIRE).*|^BT$'                     : 'Bank Transfer', 
        r'.*CREDIT.*|.*VISA.*|.*MC.*|^CC$'          : 'Credit Card', 
        r'.*DEBIT.*|.*ATM.*'                        : 'Debit Card', 
        r'^CARD$'                                   : 'Unknown Card' 
    }

    df["payment_method_clean"] = clean_source.replace(mapping, regex=True)
    valid_categories = [
        'PayPal', 'Apple Pay', 'GCash', 'Cash on Delivery', 
        'Bank Transfer', 'Credit Card', 'Debit Card', 'Unknown Card'
    ]
    
    df["payment_method_clean"] = np.where(
        df["payment_method_clean"].isin(valid_categories), 
        df["payment_method_clean"], 
        np.nan
    )
    
    return df[['payment_method_raw', 'payment_method_clean']]



def fixOrderStatus(df):
    clean_source = df["order_status_raw"].str.upper().str.strip()
    mapping = {
        r'^(DELIV.*|COMPLET.*|OK)$'             : 'Completed', 
        r'.*(SHIP.*|TRANSIT|WAY).*'             : 'Shipped', 
        r'.*(PEND.*|HOLD|AWAIT).*'              : 'Pending', 
        r'^(CANC.*|CNCLD|VOID)$'                : 'Cancelled', 
        r'.*(FAIL.*|DECLIN.*).*'                : 'Failed', 
        r'.*(RETURN.*|RMA|REFUND).*'            : 'Returned/Refunded'
    }
    
    print(f"dict: {mapping} \n")
    df["order_status_clean"] = clean_source.replace(mapping, regex=True)
    return df[['order_status_raw', 'order_status_clean']]
    


def fixReturnFlag(df):
    clean_source = df["returned_flag_raw"].astype(str).str.upper().str.strip()
    mapping = {
        r'^(Y|YES|TRUE|1|RETURNED)$'            : 'RETURNED',
        r'^(N|NO|FALSE|0|NOT\s*RETURNED)$'      : '0'
    }

    print(f"dict: {mapping} \n")
    df["returned_flag_clean"] = clean_source.replace(mapping, regex=True)
    flag_numeric = pd.to_numeric(df["returned_flag_clean"], errors='coerce')
    df["returned_flag_clean"] = flag_numeric.astype('Int64')
    return df[['returned_flag_raw', 'returned_flag_clean']]



def fixSatScore(df):
    mapping = {'excellent': '5', 'bad': '1'}
    cleaned_text = df["satisfaction_score_raw"].astype(str).str.lower().replace(mapping)
    extracted_scores = cleaned_text.str.extract(r'(\d+)', expand=False)
    scores = pd.to_numeric(extracted_scores, errors='coerce')
    
    invalid_condition = (scores < 1) | (scores > 5)
    df["satisfaction_score_clean"] = np.where(invalid_condition, np.nan, scores)
    return df[['satisfaction_score_raw', 'satisfaction_score_clean']]



def fixLoyaltyPoints(df):
    points = pd.to_numeric(df["loyalty_points_raw"], errors='coerce')
    invalid_condition = (points < 0) | (points > 100000)
    df["loyalty_points_clean"] = np.where(invalid_condition, np.nan, points)
    df["loyalty_points_clean"] = df["loyalty_points_clean"].astype('Int64')
    return df[['loyalty_points_raw', 'loyalty_points_clean']]



def fixSiteVisits(df):
    siteVisits = pd.to_numeric(df["site_visits_last30_raw"], errors='coerce')
    df["site_visits_last30_clean"] = siteVisits
    df["site_visits_last30_clean"] = df["site_visits_last30_clean"].astype('Int64')
    return df[['site_visits_last30_raw', 'site_visits_last30_clean']]



def fixSupportTickets(df):
    edge_cases = {'many': '10', 'a lot': '10'}
    cleaned_text = df["support_tickets_90d_raw"].astype(str).str.lower().replace(edge_cases)
    
    supportTickets = pd.to_numeric(cleaned_text, errors='coerce')
    df["support_tickets_90d_clean"] = np.where(supportTickets < 0, np.nan, supportTickets)
    df["support_tickets_90d_clean"] = df["support_tickets_90d_clean"].astype('Int64')
    return df[['support_tickets_90d_raw', 'support_tickets_90d_clean']]



def fixWarehouseDistance(df):
    warehouseDistance = pd.to_numeric(df["distance_to_warehouse_km_raw"], errors='coerce')
    invalid_dist = (warehouseDistance < 0) | (warehouseDistance > 20000)
    df["distance_to_warehouse_km_clean"] = np.where(invalid_dist, np.nan, warehouseDistance)
    return df[['distance_to_warehouse_km_raw', 'distance_to_warehouse_km_clean']]



def fixCustomerSegment(df):
    clean_source = df["customer_segment_raw"].astype(str).str.upper().str.strip()

    mapping = {
        r'^N$|.*NEW.*|.*FIRST.*': 'New', 
        r'.*OCC.*': 'Occasional', 
        r'.*REGULAR.*|.*REPEAT.*|.*LOYAL.*': 'Loyal', 
        r'.*PREMIUM.*|.*V\.?I\.?P.*': 'VIP', 
        r'.*DORMANT.*|.*CHURN.*|.*RISK.*': 'At Risk' 
    }
    
    # 3. Create the NEW clean column
    df["customer_segment_clean"] = clean_source.replace(mapping, regex=True)
    
    # 4. The Safety Net: Filter out junk text and catch NaNs
    valid_categories = ['New', 'Occasional', 'Loyal', 'VIP', 'At Risk']
    
    df["customer_segment_clean"] = np.where(
        df["customer_segment_clean"].isin(valid_categories), 
        df["customer_segment_clean"], 
        np.nan
    )
    
    return df[['customer_segment_raw', 'customer_segment_clean']]



In [5]:
## DEBUG WINDOW
df['returned_flag_raw'].unique()

<StringArray>
[          'No',          'Yes',         'TRUE',            'Y',
            nan,            '1',     'Returned',            'N',
        'FALSE',            '0',     'returned', 'not returned']
Length: 12, dtype: str

In [ ]:
# CHECK MISSING TOOLS
    
def missingCustomerEmail(df):
    missing_customer_email = df[df['customer_email'].isna()]
    return missing_customer_email[['record_id', 'customer_email']]

def missingDeliveryDate(df):
    missing_delivery_date = df[df['delivery_date_raw'].isna()]
    return missing_delivery_date[['record_id', 'delivery_date_clean']]

def missingSupportTicketDate(df):
    missing_support_ticket = df[df['support_ticket_date_raw'].isna()]
    return missing_support_ticket[['record_id', 'support_ticket_date_clean']]

def missingReturnReason(df):
    missing_return_reason = df[df['return_reason_raw'].isna()]
    return missing_return_reason[['record_id', 'return_reason_clean']]


# specifyWhere functions as a checker which indexes have null values in the column
# usage: specifyWhere(pandas.DataFrame, 'column_name')
def specifyWhere(df, columns):
    missing_locations = {}
    for col in columns:
        null_indeces = df[df[col].isnull()].index.tolist()

        if null_indeces:
            missing_locations[col] = null_indeces

    return missing_locations


In [ ]:
## PART 3

## FIXME: 
def signupDate(df, record_id):
   return df.loc[df['signup_date_raw']].at[record_id]

def order_datetime(df, date):
   return df.loc[df['order_date_raw'] == date] 

def order_datetime(df, date):
   return df.loc[df['order_date_raw'] == date] 

def support_ticket_datetime(df, date):
   return df.loc[df['support_ticket_date_raw'] == date] 

def days_from_signup_to_order(df, date):
   ## pull signup date of a specific customer
   return 0

def shipping_delay_days(df, date):
   return df.loc[df['order_date_raw'] == date] 

# /FIXME







In [ ]:
## PART 5


    


In [ ]:
## PART 6


def fixDeliverySequence(df):
    order_date = pd.to_datetime(df["order_date_raw"], format='mixed', errors='coerce')
    delivery_date = pd.to_datetime(df["delivery_date_raw"], format='mixed', errors='coerce')
    invalid_condition = order_date > delivery_date
    
    df["order_date_clean"] = order_date
    df["delivery_date_clean"] = np.where(invalid_condition, pd.NaT, delivery_date)

    ## additional check
    duration = (df["delivery_date_clean"] - df["order_date_clean"]).dt.days
    df["delivery_duration_clean"] = duration.astype('Int64')
    
    return df[['order_date_raw', 'delivery_date_raw', 'delivery_date_clean', 'delivery_duration_clean']]




















def errorLog(df):
    audit_columns = [
        ('quantity_raw', 'quantity_clean'),
        ('unit_price_raw', 'unit_price_clean'),
        ('satisfaction_score_raw', 'satisfaction_score_clean'),
        ('loyalty_points_raw', 'loyalty_points_clean'),
        ('distance_to_warehouse_km_raw', 'distance_to_warehouse_km_clean'),
        ('site_visits_last30_raw', 'site_visits_last30_clean'),
        ('support_tickets_90d_raw', 'support_tickets_90d_clean'),
        ('shipping_cost_raw', 'shipping_cost_clean'),
        ('order_date_raw', 'delivery_duration_clean'),
        ('payment_method_raw', 'payment_method_clean'),
        ('customer_segment_raw', 'customer_segment_clean')
    ]
    
    error_list = []
    for raw_col, clean_col in audit_columns:
        is_rejected = df[clean_col].isna() & df[raw_col].notna() & (df[raw_col] != '')
        bad_rows = df[is_rejected][['record_id', raw_col]].copy()
        bad_rows = bad_rows.rename(columns={raw_col: 'invalid_value'})
        bad_rows['source_column'] = raw_col
        error_list.append(bad_rows)

    if error_list:
        error_log = pd.concat(error_list, ignore_index=True)
        return error_log[['record_id', 'source_column', 'invalid_value']]
    
    else:
        print("No errors found! Dataset is perfectly clean.")
        return pd.DataFrame()

In [ ]:
## Clean Everything
def clean(df):
    fixGender(df)
    fixSource(df)
    fixCountry(df)
    fixTime(df)
    fixEncoding(df)
    fixCategory(df)
    fixDiscount(df)
    fixDeliverySequence(df)
    fixQuantity(df)
    fixUnitPrice(df)
    fixShippingCost(df)
    fixItemWeight(df)
    fixSatScore(df)
    fixLoyaltyPoints(df)
    fixSiteVisits(df)
    fixSupportTickets(df)
    fixWarehouseDistance(df)
    fixCustomerSegment(df)
    fixPaymentMethod(df)
    raw_columns = [col for col in df.columns if col.endswith('_raw')]
    df_final = df.drop(columns=raw_columns)
    
    return df_final


## COMMANDS
# clean(df) cleans data [run first]
# errors = errorLog(df)
# display(errors) displays all errors [run last]

In [ ]:
df_clean = clean(df)
errors = errorLog(df)

In [ ]:
display(errors)

In [ ]:
# DEBUG
# df[['record_id', 'source_system', 'customer_id', 'customer_name', 'customer_email', 'phone_raw']]
# df[['age_raw', 'gender_raw', 'signup_date_raw', 'order_id', 'order_date_raw']]



In [ ]:
# PART 7

cols_to_scale = [
    'unit_price_clean', 'quantity_clean', 'shipping_cost_clean', 
    'item_weight_clean', 'loyalty_points_clean', 'site_visits_last30_clean', 
    'support_tickets_90d_clean', 'distance_to_warehouse_km_clean', 'delivery_duration_clean'
]

std_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()
robust_scaler = RobustScaler()

std_cols = [f"{col}_std" for col in cols_to_scale]
df[std_cols] = std_scaler.fit_transform(df[cols_to_scale])

minmax_cols = [f"{col}_minmax" for col in cols_to_scale]
df[minmax_cols] = minmax_scaler.fit_transform(df[cols_to_scale])

robust_cols = [f"{col}_robust" for col in cols_to_scale]
df[robust_cols] = robust_scaler.fit_transform(df[cols_to_scale])

print(df[['unit_price_clean', 'unit_price_clean_std', 'unit_price_clean_minmax', 'unit_price_clean_robust']].head())


1. Which scaling method is most affected by outliers?

Min-Max Normalization. This scaler uses the absolute minimum and absolute maximum values of the dataset to squish everything into a 0-to-1 range. If you have just one massive outlier, it becomes the 1, and all of your normal data gets crushed into a tiny, indistinguishable cluster near 0. (Note: Standardization is also negatively affected by outliers because they pull the mean, but Min-Max is completely destroyed by them).

2. Which method is best for skewed data?

Robust Scaling. Robust scaling ignores the mean, minimum, and maximum entirely. Instead, it uses the Median and the Interquartile Range (IQR)—which is just the middle 50% of your data. Because it completely ignores the long tails of skewed data (and extreme outliers), it scales the bulk of your data perfectly without being distorted.

3. Which variables should not be scaled?

Categorical / Binary flags: Columns that are just 0 or 1 (like is_premium_member).
Ordinal scores with strict boundaries: In your dataset, satisfaction_score_clean (which operates on a 1-to-5 star rating) usually shouldn't be scaled. It already has a strict, highly interpretable mathematical boundary.
Identifiers: Things like record_id or customer_id are arbitrary labels, not math, and should never be scaled.

4. Why should scaling be done after cleaning and imputation?

Scalers rely on mathematical absolute truths about your column (the Mean, Median, Min, Max, and Standard Deviation).
If you scale before cleaning out a 999 placeholder quantity, your scaler uses 999 as the maximum, ruining the scale.
If you scale before imputing missing values, and then try to fill in blanks later, you are either injecting raw, unscaled numbers into a scaled column (which breaks your machine learning model), or you are forcing the imputer to calculate the mean of an incomplete, skewed distribution. Always clean, then impute, then scale.

In [ ]:
analysis_cols = [
    'unit_price_clean',
    'quantity_clean',
    'shipping_cost_clean',
    'discount_clean',
    'item_weight_clean',
    'loyalty_points_clean',
    'site_visits_last30_clean',
    'support_tickets_90d_clean',
    'satisfaction_score_clean',
    'delivery_duration_clean'
]

summary_stats = df[analysis_cols].describe()
display(summary_stats.T)

skewness = df[analysis_cols].skew().sort_values(ascending=False)
print("\nSkewness:")
print(skewness)

1. ``unit_price_clean`` is heavily right-skewed with extreme outliers
Median is 185.65 but max is 256,397. The mean (3,699) is 20x the median, pulled entirely by a handful of high-value orders. These are likely wholesale or enterprise transactions mixed into what appears to be mostly retail data.

2. ``quantity_clean`` orders are overwhelmingly small
75% of orders have a quantity of 3 or less, yet the max is 100. The extreme skew (10.5) confirms a small number of bulk orders are distorting the distribution. Typical customer buys 1–2 items.

3. ``support_tickets_90d_clean`` shows most customers have zero issues
Median and 75th percentile are both 0–1, meaning the majority of customers never raise a support ticket. The max of 35 suggests a small group of highly dissatisfied or high-maintenance customers.

4. ``satisfaction_score_clean`` is the only normally distributed variable
Skew of 0.01 is essentially zero. Mean (3.0) and median (3.0) are identical, scores range cleanly 1–5. This is the only column where StandardScaler would work well without any preprocessing.

5. ``delivery_duration_clean`` has a suspicious minimum of 0
A delivery duration of 0 days means the order was delivered the same day it was placed, which is plausible but worth flagging. The wide range (0–356 days) with a std of 41 suggests delivery performance is highly inconsistent across regions or fulfillment methods.

In [ ]:
## PART 8 - Sales Performance Analysis

# business metrics
df['gross_sales'] = df['unit_price_clean'] * df['quantity_clean']
df['discount_amount'] = df['gross_sales'] * (df['discount_clean'] / 100)
df['net_sales'] = df['gross_sales'] - df['discount_amount']
df['estimated_total_order_value'] = df['net_sales'] + df['shipping_cost_clean']

# summary metrics
total_gross_sales = df['gross_sales'].sum()
total_net_sales   = df['net_sales'].sum()
avg_order_value   = df['estimated_total_order_value'].mean()

print(f"Total Gross Sales: ${total_gross_sales:,.2f}")
print(f"Total Net Sales: ${total_net_sales:,.2f}")
print(f"Average Order Value: ${avg_order_value:,.2f}")

# sales by product categoy
sales_by_category = (
    df.groupby('product_category_raw')['net_sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'product_category_raw': 'category', 'net_sales': 'total_net_sales'})
)
display(sales_by_category)

# sales by country
sales_by_country = (
    df.groupby('country_raw')['net_sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'country_raw': 'country', 'net_sales': 'total_net_sales'})
)
display(sales_by_country)

# sales by payment method
sales_by_payment = (
    df.groupby('payment_method_clean')['net_sales']
    .sum()
    .sort_values(ascending=False)
    .reset_index()
    .rename(columns={'payment_method_rclean': 'payment_method', 'net_sales': 'total_net_sales'})
)
display(sales_by_payment)

# monthly sales trend
df['order_month'] = pd.to_datetime(df['order_date_raw'], errors='coerce').dt.to_period('M')
monthly_sales = (
    df.groupby('order_month')['net_sales']
    .sum()
    .reset_index()
)
monthly_sales['order_month'] = monthly_sales['order_month'].astype(str)
display(monthly_sales)

In [ ]:
# sales by product category
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(sales_by_category['category'], sales_by_category['total_net_sales'])
ax.set_title('Net Sales by Product Category')
ax.set_xlabel('Category')
ax.set_ylabel('Net Sales ($)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

#sales by country
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(sales_by_country['country'], sales_by_country['total_net_sales'])
ax.set_title('Net Sales by Country')
ax.set_xlabel('Country')
ax.set_ylabel('Net Sales ($)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

# monthly sales trend
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_sales['order_month'], monthly_sales['net_sales'], marker='o')
ax.set_title('Monthly Sales Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Net Sales ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

categories = df['product_category_raw'].dropna().unique()
data_by_category = [
    df[df['product_category_raw'] == cat]['estimated_total_order_value'].dropna()
    for cat in categories
]

# boxplot by category
fig, ax = plt.subplots(figsize=(12, 6))
ax.boxplot(data_by_category, labels=categories, showfliers=False) # showfliers=False hides extreme outliers for readability
ax.set_title('Order Value by Product Category')
ax.set_xlabel('Category')
ax.set_ylabel('Estimated Total Order Value ($)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
## PART 9 - Customer Behavior Analysis

# features 
df['days_from_signup_to_order'] = (
    pd.to_datetime(df['order_date_raw'], errors='coerce') - 
    pd.to_datetime(df['signup_date_raw'], errors='coerce')
).dt.days

# site visits vs purchase quality
visit_quantity_corr = df[['site_visits_last30_clean', 'quantity_clean']].corr()
print("Site Visits vs Quantity Correlation:")
print(visit_quantity_corr)

# loyalty points vs order value
loyalty_order_corr = df[['loyalty_points_clean', 'estimated_total_order_value']].corr()
print("\nLoyalty Points vs Order Value Correlation:")
print(loyalty_order_corr)

# average order value
aov_by_segment = (
    df.groupby('customer_segment_clean')['estimated_total_order_value']
    .agg(avg_order_value='mean', count='count')
    .sort_values('avg_order_value', ascending=False)
    .reset_index()
)
display(aov_by_segment)

# support ticket frequency
tickets_by_segment = (
    df.groupby('customer_segment_clean')['support_tickets_90d_clean']
    .agg(avg_tickets='mean', total_tickets='sum')
    .sort_values('avg_tickets', ascending=False)
    .reset_index()
)
display(tickets_by_segment)

# repeat customer behavior
repeat_customers = (
    df.groupby('customer_id')
    .agg(
        order_count=('order_id', 'count'),
        total_spent=('estimated_total_order_value', 'sum'),
        avg_order_value=('estimated_total_order_value', 'mean')
    )
    .sort_values('order_count', ascending=False)
    .reset_index()
)

repeat_rate = (repeat_customers['order_count'] > 1).sum() / len(repeat_customers) * 100
print(f"\nRepeat Customer Rate: {repeat_rate:.2f}%")
display(repeat_customers.head(10))

In [ ]:
# plots
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# site visits vs quantity
axes[0, 0].scatter(
    df['site_visits_last30_clean'], df['quantity_clean'],
    alpha=0.3, edgecolors='none'
)
axes[0, 0].set_title('Site Visits vs Purchase Quantity')
axes[0, 0].set_xlabel('Site Visits (last 30 days)')
axes[0, 0].set_ylabel('Quantity')

# loyalty points vs order value
axes[0, 1].scatter(
    df['loyalty_points_clean'], df['estimated_total_order_value'],
    alpha=0.3, edgecolors='none'
)
axes[0, 1].set_title('Loyalty Points vs Order Value')
axes[0, 1].set_xlabel('Loyalty Points')
axes[0, 1].set_ylabel('Estimated Total Order Value ($)')

# average order by segment
axes[1, 0].bar(aov_by_segment['customer_segment_clean'], aov_by_segment['avg_order_value'])
axes[1, 0].set_title('Avg Order Value by Customer Segment')
axes[1, 0].set_xlabel('Segment')
axes[1, 0].set_ylabel('Avg Order Value ($)')
plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=30, ha='right')

# avg support tickets by segment
axes[1, 1].bar(tickets_by_segment['customer_segment_clean'], tickets_by_segment['avg_tickets'])
axes[1, 1].set_title('Avg Support Tickets by Customer Segment')
axes[1, 1].set_xlabel('Segment')
axes[1, 1].set_ylabel('Avg Tickets (90 days)')
plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=30, ha='right')

plt.tight_layout()
plt.show()

In [ ]:
# PART 10 - Export

# save all plots before export
fig_category, ax = plt.subplots(figsize=(10, 5))
ax.bar(sales_by_category['category'], sales_by_category['total_net_sales'])
ax.set_title('Net Sales by Product Category')
ax.set_xlabel('Category')
ax.set_ylabel('Net Sales ($)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
fig_category.savefig('plot_net_sales_by_category.png', dpi=150)
plt.show()

fig_country, ax = plt.subplots(figsize=(10, 5))
ax.bar(sales_by_country['country'], sales_by_country['total_net_sales'])
ax.set_title('Net Sales by Country')
ax.set_xlabel('Country')
ax.set_ylabel('Net Sales ($)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
fig_country.savefig('plot_net_sales_by_country.png', dpi=150)
plt.show()

fig_monthly, ax = plt.subplots(figsize=(12, 5))
ax.plot(monthly_sales['order_month'], monthly_sales['net_sales'], marker='o')
ax.set_title('Monthly Sales Trend')
ax.set_xlabel('Month')
ax.set_ylabel('Net Sales ($)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
fig_monthly.savefig('plot_monthly_sales_trend.png', dpi=150)
plt.show()

categories = df['product_category_raw'].dropna().unique()
data_by_category = [
    df[df['product_category_raw'] == cat]['estimated_total_order_value'].dropna()
    for cat in categories
]
fig_box, ax = plt.subplots(figsize=(12, 6))
ax.boxplot(data_by_category, labels=categories, showfliers=False)
ax.set_title('Order Value by Product Category')
ax.set_xlabel('Category')
ax.set_ylabel('Estimated Total Order Value ($)')
plt.xticks(rotation=30, ha='right')
plt.tight_layout()
fig_box.savefig('plot_order_value_boxplot.png', dpi=150)
plt.show()

fig_behavior, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].scatter(df['site_visits_last30_clean'], df['quantity_clean'], alpha=0.3, edgecolors='none')
axes[0, 0].set_title('Site Visits vs Purchase Quantity')
axes[0, 0].set_xlabel('Site Visits (last 30 days)')
axes[0, 0].set_ylabel('Quantity')
axes[0, 1].scatter(df['loyalty_points_clean'], df['estimated_total_order_value'], alpha=0.3, edgecolors='none')
axes[0, 1].set_title('Loyalty Points vs Order Value')
axes[0, 1].set_xlabel('Loyalty Points')
axes[0, 1].set_ylabel('Estimated Total Order Value ($)')
axes[1, 0].bar(aov_by_segment['customer_segment_clean'], aov_by_segment['avg_order_value'])
axes[1, 0].set_title('Avg Order Value by Customer Segment')
axes[1, 0].set_xlabel('Segment')
axes[1, 0].set_ylabel('Avg Order Value ($)')
plt.setp(axes[1, 0].xaxis.get_majorticklabels(), rotation=30, ha='right')
axes[1, 1].bar(tickets_by_segment['customer_segment_clean'], tickets_by_segment['avg_tickets'])
axes[1, 1].set_title('Avg Support Tickets by Customer Segment')
axes[1, 1].set_xlabel('Segment')
axes[1, 1].set_ylabel('Avg Tickets (90 days)')
plt.setp(axes[1, 1].xaxis.get_majorticklabels(), rotation=30, ha='right')
plt.tight_layout()
fig_behavior.savefig('plot_customer_behavior.png', dpi=150)
plt.show()

print("All plots saved.")

# export clean dataframe
export_cols = [col for col in df.columns if not col.endswith('_raw')]
df[export_cols].to_csv('ecommerce_data_final.csv', index=False)

print(f"Export complete. Shape: {df[export_cols].shape}")